# SNU AI Challenge - Basic CV Augmented 5-Fold Experiment

This experiment uses the new MD5 group-stratified folds. It reuses the 9,535 split-independent original frozen representations, extracts only the 6,445 basic-CV augmented rows with batched GPU forwards, validates on unmodified original rows only, and writes new checkpoints to a separate Drive result root.

## 1. Configuration and mandatory GPU confirmation

Before setting `GPU_CONFIG_CONFIRMED = True`, confirm the detected GPU, VRAM, feature batch size, image worker count, and training batch size with the user. Start with feature batch 32 on an A100 40 GB; increase only after the GPU telemetry log shows headroom.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai_augmented_v1"))
GROUP_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/group_stratified_folds_5.csv"
DRIVE_GROUP_FOLDS_CSV = DRIVE_ROOT / "splits/group_stratified_folds_5.csv"
FULL_BUNDLE = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
AUGMENTED_BUNDLE = DRIVE_ROOT / "snu_colab_basic_cv_augmented_bundle.tar"
CODE_BUNDLE = DRIVE_ROOT / "snu_colab_augmented_training_code.tar"
RAW_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
AUGMENTED_CACHE_SHARDS = DRIVE_ROOT / "cache/basic_cv_augmented_6445_batched_v1"
RESULTS_ROOT = DRIVE_ROOT / "results/basic_cv_augmented_group_stratified_v1"
COMBINED_CACHE_ROOT = WORKDIR / "feature_cache/basic_cv_augmented_combined"

GPU_CONFIG_CONFIRMED = True
FEATURE_BATCH_SIZE = 32
IMAGE_WORKERS = 16
TRAINING_BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
FOLDS_TO_RUN = [0, 1, 2, 3, 4]


## 2. Mount Drive, extract bundles, and build the MD5 group-stratified split

In [ ]:
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.13.0", "accelerate==1.14.0", "pillow>=10"], check=True)
for path in (FULL_BUNDLE, AUGMENTED_BUNDLE, CODE_BUNDLE):
    if not path.is_file():
        raise FileNotFoundError(path)
WORKDIR.mkdir(parents=True, exist_ok=True)
if (not (WORKDIR / "snuaichallenge_data/train.csv").is_file()
        or not (WORKDIR / "snuaichallenge_data/train").is_dir()):
    subprocess.run(["tar", "-xf", str(FULL_BUNDLE), "-C", str(WORKDIR)], check=True)
if not (WORKDIR / "augmentation/generated/basic_cv/train_augmented.csv").is_file():
    subprocess.run(["tar", "-xf", str(AUGMENTED_BUNDLE), "-C", str(WORKDIR)], check=True)
subprocess.run(["tar", "-xf", str(CODE_BUNDLE), "-C", str(WORKDIR)], check=True)
import ast
import hashlib
import json
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
GROUP_SPLIT_SEED = 42
GROUP_SPLIT_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3
GROUP_SPLIT_METHOD = "md5_stratified_group_kfold_v1"


def _compute_md5(path):
    with path.open("rb") as file:
        return hashlib.md5(file.read()).hexdigest()


def _build_md5_group_stratified_folds(train_csv, image_root):
    train = pd.read_csv(train_csv)
    train["answer_tuple"] = train["Answer"].map(
        lambda value: str(ast.literal_eval(value))
    )

    image_records = []
    for _, row in train.iterrows():
        for image_index in range(1, 5):
            image_records.append(
                {
                    "sample_id": row["Id"],
                    "path": image_root / str(row["Id"]) / str(row[f"Input_{image_index}"]),
                }
            )

    print(f"Computing MD5 for {len(image_records)} training images...")
    with ThreadPoolExecutor(max_workers=32) as executor:
        hashes = list(executor.map(_compute_md5, [record["path"] for record in image_records]))
    for record, image_hash in zip(image_records, hashes, strict=True):
        record["hash"] = image_hash

    hash_counter = defaultdict(set)
    for record in image_records:
        hash_counter[record["hash"]].add(record["sample_id"])
    hash_to_samples = {
        image_hash: sample_ids
        for image_hash, sample_ids in hash_counter.items()
        if len(sample_ids) <= MAX_SAMPLES_PER_HASH
    }

    graph = nx.Graph()
    graph.add_nodes_from(train["Id"].unique())
    for sample_ids in hash_to_samples.values():
        sample_ids = list(sample_ids)
        for left in range(len(sample_ids)):
            for right in range(left + 1, len(sample_ids)):
                graph.add_edge(sample_ids[left], sample_ids[right])

    components = list(nx.connected_components(graph))
    id_to_cluster = {
        sample_id: cluster_id
        for cluster_id, component in enumerate(components)
        for sample_id in component
    }
    train["cluster_id"] = train["Id"].map(id_to_cluster)

    splitter = StratifiedGroupKFold(
        n_splits=GROUP_SPLIT_FOLDS,
        shuffle=True,
        random_state=GROUP_SPLIT_SEED,
    )
    train["Fold"] = -1
    for fold, (_, validation_indices) in enumerate(
        splitter.split(
            train,
            train["answer_tuple"],
            groups=train["cluster_id"],
        )
    ):
        train.loc[validation_indices, "Fold"] = fold

    train["split_method"] = GROUP_SPLIT_METHOD
    train["split_seed"] = GROUP_SPLIT_SEED
    train["split_max_samples_per_hash"] = MAX_SAMPLES_PER_HASH
    train["split_sklearn_version"] = sklearn.__version__
    build_summary = {
        "clusters": len(components),
        "excluded_frequent_hashes": sum(
            len(sample_ids) > MAX_SAMPLES_PER_HASH
            for sample_ids in hash_counter.values()
        ),
    }
    return train, build_summary


def _validate_md5_group_stratified_folds(folded, raw_train):
    required = {
        "Id", "Answer", "No_ordering", "answer_tuple", "cluster_id", "Fold",
        "split_method", "split_seed", "split_max_samples_per_hash",
        "split_sklearn_version",
    }
    missing = required.difference(folded.columns)
    if missing:
        raise ValueError(f"Group split is missing columns: {sorted(missing)}")
    if folded["Id"].astype(str).tolist() != raw_train["Id"].astype(str).tolist():
        raise ValueError("Group split IDs/order do not match train.csv")
    if folded["Id"].duplicated().any():
        raise ValueError("Group split contains duplicate IDs")

    expected_answers = raw_train["Answer"].map(lambda value: str(ast.literal_eval(value)))
    if folded["answer_tuple"].astype(str).tolist() != expected_answers.tolist():
        raise ValueError("Group split Answer labels do not match train.csv")
    if set(folded["split_method"].astype(str)) != {GROUP_SPLIT_METHOD}:
        raise ValueError("Unexpected group split method")
    if set(folded["split_seed"].astype(int)) != {GROUP_SPLIT_SEED}:
        raise ValueError("Unexpected group split seed")
    if set(folded["split_max_samples_per_hash"].astype(int)) != {MAX_SAMPLES_PER_HASH}:
        raise ValueError("Unexpected MD5 reuse threshold")

    folded["Fold"] = folded["Fold"].astype(int)
    if set(folded["Fold"]) != set(range(GROUP_SPLIT_FOLDS)):
        raise ValueError("Expected folds 0 through 4")
    cross_fold_clusters = int(
        (folded.groupby("cluster_id")["Fold"].nunique() > 1).sum()
    )
    if cross_fold_clusters:
        raise ValueError(f"{cross_fold_clusters} MD5 clusters cross folds")

    identity_mask = folded["answer_tuple"].eq("[1, 2, 3, 4]")
    no_ordering_mask = folded["No_ordering"].astype(str).str.lower().eq("true")
    if not identity_mask.equals(no_ordering_mask):
        raise ValueError("Identity labels and No_ordering=True rows differ")
    classes_per_fold = folded.groupby("Fold")["answer_tuple"].nunique().to_dict()
    if set(classes_per_fold.values()) != {24}:
        raise ValueError(f"Every fold must contain all 24 classes: {classes_per_fold}")

    return {
        "method": GROUP_SPLIT_METHOD,
        "rows": len(folded),
        "classes": folded["answer_tuple"].nunique(),
        "identity_rows": int(identity_mask.sum()),
        "clusters": folded["cluster_id"].nunique(),
        "cross_fold_clusters": cross_fold_clusters,
        "fold_counts": {
            str(fold): int(count)
            for fold, count in folded["Fold"].value_counts().sort_index().items()
        },
        "built_with_sklearn": str(folded["split_sklearn_version"].iloc[0]),
    }


GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
DRIVE_GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_GROUP_FOLDS_CSV.is_file():
    shutil.copy2(DRIVE_GROUP_FOLDS_CSV, GROUP_FOLDS_CSV)
elif not GROUP_FOLDS_CSV.is_file():
    folded_frame, group_split_build = _build_md5_group_stratified_folds(
        TRAIN_CSV,
        IMAGE_ROOT,
    )
    temporary_csv = GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    folded_frame.to_csv(temporary_csv, index=False)
    temporary_csv.replace(GROUP_FOLDS_CSV)
    print("group split build:", json.dumps(group_split_build, indent=2))

raw_train_frame = pd.read_csv(TRAIN_CSV)
group_fold_frame = pd.read_csv(GROUP_FOLDS_CSV)
group_split_summary = _validate_md5_group_stratified_folds(
    group_fold_frame,
    raw_train_frame,
)
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    temporary_drive_csv = DRIVE_GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    shutil.copy2(GROUP_FOLDS_CSV, temporary_drive_csv)
    temporary_drive_csv.replace(DRIVE_GROUP_FOLDS_CSV)
print("group split:", json.dumps(group_split_summary, indent=2))

AUGMENTED_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(WORKDIR, AUGMENTED_CACHE_SHARDS, RESULTS_ROOT, sep="\n")


## 3. Confirm GPU settings with the user

Run this inspection cell, discuss the output, then change `GPU_CONFIG_CONFIRMED` to `True` in the configuration cell. The experiment runner refuses to start otherwise.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
props = torch.cuda.get_device_properties(0)
print({"gpu": props.name, "vram_gib": props.total_memory / 1024**3, "feature_batch": FEATURE_BATCH_SIZE, "image_workers": IMAGE_WORKERS, "training_batch": TRAINING_BATCH_SIZE, "evaluation_batch": EVALUATION_BATCH_SIZE})
if not GPU_CONFIG_CONFIRMED:
    raise RuntimeError("Confirm these GPU settings with the user before caching or training.")


## 4. Cache only the augmented 6,445 rows

The split-independent original representations are restored and reused; legacy fold checkpoints and metrics are not reused. Qwen and ConvNeXt process multiple samples per forward pass. Cache scans report progress every 500 samples, and feature extraction reports completed/total, percentage, remaining samples, throughput, elapsed time, and ETA after every GPU batch. GPU telemetry is saved as `feature_cache_gpu_dmon.log` under the new result root.

In [ ]:
common_command = [
    sys.executable, str(WORKDIR / "scripts/run_augmented_5fold_colab.py"),
    "--original-csv", str(WORKDIR / "snuaichallenge_data/splits/group_stratified_folds_5.csv"),
    "--original-image-root", str(WORKDIR / "snuaichallenge_data/train"),
    "--augmented-csv", str(WORKDIR / "augmentation/generated/basic_cv/train_augmented.csv"),
    "--augmented-image-root", str(WORKDIR / "augmentation/generated/basic_cv/images"),
    "--combined-cache-root", str(COMBINED_CACHE_ROOT),
    "--raw-cache-shards", str(RAW_CACHE_SHARDS),
    "--augmented-cache-shards", str(AUGMENTED_CACHE_SHARDS),
    "--results-root", str(RESULTS_ROOT),
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--image-workers", str(IMAGE_WORKERS),
    "--training-batch-size", str(TRAINING_BATCH_SIZE),
    "--evaluation-batch-size", str(EVALUATION_BATCH_SIZE),
    "--folds", *map(str, FOLDS_TO_RUN),
    "--confirm-gpu-config",
]
subprocess.run([*common_command, "--cache-only"], cwd=WORKDIR, check=True)


## 5. Run leakage-safe five-fold training and one final scorer

Each fold validates only on its unmodified original rows. An augmented row is included in training only when its source original is outside the validation fold. The five fold checkpoints are validation artifacts and must not be ensembled.

In [ ]:
subprocess.run(common_command, cwd=WORKDIR, check=True)


## Expected isolated outputs

- `MyDrive/snu ai/cache/basic_cv_augmented_6445_batched_v1/`: augmented-only cache shards and manifest
- `MyDrive/snu ai/results/basic_cv_augmented_group_stratified_v1/fold_*/`: five validation artifacts
- `MyDrive/snu ai/results/basic_cv_augmented_group_stratified_v1/five_fold_summary.json`: original-only validation result comparable to the new group-stratified baseline
- `MyDrive/snu ai/results/basic_cv_augmented_group_stratified_v1/final_scorer/final.pt`: one final model trained on 9,535 originals plus 6,445 augmented rows

After completion, record the measured result and likely cause in `MODELS.md`.